In [1]:
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from scipy.spatial.distance import cdist

from sklearn.cluster import KMeans
from sklearn.cluster import MiniBatchKMeans
from sklearn.cluster import BisectingKMeans
from sklearn.cluster import Birch
from sklearn.cluster import MeanShift
from sklearn.cluster import estimate_bandwidth
from sklearn.mixture import GaussianMixture
from sklearn.cluster import DBSCAN

from sklearn.pipeline import (
    Pipeline,
    TransformerMixin
)

from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder
)

from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

from sklearn.model_selection import (
    train_test_split,
    cross_validate,
    StratifiedKFold,
    ShuffleSplit
)

from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import GridSearchCV

# O notebook roda a partir de `notebooks/`; a raiz entra no path para que
# `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.data import dicionario as dic


In [2]:
data_path = Path('../data/processed/imoveis_curados.parquet')
dict_path = Path('../data/processed/dicionario.csv')

In [3]:
df = pd.read_parquet(data_path)
display(Markdown(f'### Conjunto de dados — {len(df):,} imóveis × {df.shape[1]} colunas'.replace(',', '.')))
display(df.head())

df_dict = (
    pd
    .read_csv(dict_path, sep=';')
    .assign(coluna=lambda d: d['variavel'])
)
display(Markdown('### Dicionário de dados'))
display(df_dict.head())

display(Markdown(
    'Tipos declarados: '
    + ' · '.join(f'**{tipo}** {n}' for tipo, n in df_dict['Tipo'].value_counts().items())
))

### Conjunto de dados — 2.350 imóveis × 83 colunas

,ordem,pid,subclasse,zoneamento,area_terreno,tipo_viela,formato_lote,relevo,config_lote,declividade,...,preco_venda,area_varandas,banheiros_total,idade_imovel,anos_desde_reforma,tem_porao,tem_garagem,tem_lareira,tem_segundo_pavimento,preco_por_pe2
0,2,526350040,20,RH,11622,SemViela,4,Lvl,Inside,3,...,105000,260,1.0,49,49,1,1,0,0,117.187500
1,3,526351010,20,RL,14267,SemViela,3,Lvl,Corner,3,...,172000,429,1.5,52,52,1,1,0,0,129.420617
2,4,526353030,20,RL,11160,SemViela,4,Lvl,Corner,3,...,244000,0,3.5,42,42,1,1,1,0,115.639810
3,5,527105010,60,RL,13830,SemViela,3,Lvl,Inside,3,...,189900,246,2.5,13,12,1,1,1,1,116.574586
4,6,527105030,60,RL,9978,SemViela,3,Lvl,Inside,3,...,195500,396,2.5,12,12,1,1,1,1,121.882793


### Dicionário de dados

,variavel,descricao,valores,Tipo,coluna
0,Order,Número de ordem do registro no arquivo original,"[1, 2930]",Discreta,ordem
1,PID,Identificador cadastral do lote na prefeitura ...,"[526301100, 1007100110]",Discreta,pid
2,MS SubClass,Código do tipo de imóvel envolvido na venda (i...,"[20, 190]",Discreta,subclasse
3,MS Zoning,Classificação geral de zoneamento urbano do lote,"['RL', 'A (agr)']",Nominal,zoneamento
4,Lot Frontage,Testada: pés lineares de rua conectados ao lote,"[21.0, 313.0]",Contínua,testada_lote


Tipos declarados: **Discreta** 28 · **Nominal** 23 · **Ordinal** 20 · **Contínua** 11

In [4]:
# A curadoria removeu 8 colunas do dicionário e derivou 9 que não estão nele.
# A conferência fica explícita: o que não bate aqui vira decisão, não surpresa.
sem_mapa = df_dict.loc[df_dict['coluna'].isna(), 'variavel'].tolist()
fora_do_parquet = [c for c in df_dict['coluna'].dropna() if c not in df.columns]
fora_do_dicionario = [c for c in df.columns if c not in set(df_dict['coluna'].dropna())]

display(Markdown(
    f'- variáveis do dicionário sem tradução: **{len(sem_mapa)}** {sem_mapa}\n'
    f'- no dicionário, ausentes do parquet: **{len(fora_do_parquet)}** {fora_do_parquet}\n'
    f'- no parquet, ausentes do dicionário (derivadas na curadoria): '
    f'**{len(fora_do_dicionario)}** {fora_do_dicionario}'
))


- variáveis do dicionário sem tradução: **0** []
- no dicionário, ausentes do parquet: **8** ['testada_lote', 'tipo_via', 'infraestrutura', 'condicao_entorno_2', 'material_telhado', 'ano_garagem', 'area_piscina', 'qualidade_piscina']
- no parquet, ausentes do dicionário (derivadas na curadoria): **9** ['area_varandas', 'banheiros_total', 'idade_imovel', 'anos_desde_reforma', 'tem_porao', 'tem_garagem', 'tem_lareira', 'tem_segundo_pavimento', 'preco_por_pe2']

In [5]:
# Analisando o preço da venda
df.preco_venda.describe()

count      2350.000000
mean     173344.605532
std       66691.089678
min       35000.000000
25%      129050.000000
50%      157500.000000
75%      203101.250000
max      615000.000000
Name: preco_venda, dtype: float64

In [6]:
useless_columns = [
    'ordem', 'pid'
]


def colunas_do_tipo(*tipos):
    """Colunas da base curada que o dicionário declara com um dos `tipos`."""
    return [
        coluna
        for coluna in df_dict.query('Tipo in @tipos')['coluna'].dropna()
        if coluna in df.columns and coluna not in useless_columns
    ]


# Contínua e discreta
continuous_columns = colunas_do_tipo('Contínua', 'Discreta')

# Ordinal & Nominal
ordinal_columns = colunas_do_tipo('Ordinal')

nominal_columns = colunas_do_tipo('Nominal')

# As devirações
derived_continuous = ['area_varandas', 'banheiros_total', 'idade_imovel', 'anos_desde_reforma']
derived_binary = ['tem_porao', 'tem_garagem', 'tem_lareira', 'tem_segundo_pavimento']
continuous_columns = continuous_columns + derived_continuous + derived_binary

# Impressões
display(Markdown(
    f'- contínuas e discretas: **{len(continuous_columns)}**\n'
    f'- ordinais: **{len(ordinal_columns)}**\n'
    f'- nominais: **{len(nominal_columns)}**\n'
    f'- fora da modelagem: **{len(useless_columns)}** {useless_columns}'
))


- contínuas e discretas: **42**
- ordinais: **18**
- nominais: **20**
- fora da modelagem: **2** ['ordem', 'pid']

In [7]:
continuous_preprocessor = Pipeline(steps=[
    ('normalization', MinMaxScaler()) # Normalização
])

ordinal_preprocessor = Pipeline(steps=[
    ('normalization', MinMaxScaler()) # Normalização — preserva a ordem e iguala o peso
])

nominal_preprocessor = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore')), # Codificação de variáveis
])

preprocessor = ColumnTransformer(transformers=[
    ('continuous', continuous_preprocessor, continuous_columns),
    ('ordinal', ordinal_preprocessor, ordinal_columns),
    ('nominal', nominal_preprocessor, nominal_columns)
])


In [8]:
X = df.drop(columns=useless_columns)
y = None # Aprendizagem não supervisionada: o rótulo não é utilizado

# A grade de `bandwidth` do mean shift não pode ser copiada do 07: ela depende da
# escala das distâncias, e a escala mudou junto com o espaço. Uma transformação
# única (não supervisionada, sem olhar o rótulo) dá a âncora.
X_ref = preprocessor.fit_transform(X)
bandwidth_ref = {
    q: estimate_bandwidth(X_ref, quantile=q, random_state=42)
    for q in (0.1, 0.2, 0.3, 0.5)
}
distancias = cdist(X_ref[:800], X_ref[:800])[np.triu_indices(800, k=1)]

display(Markdown(
    f'Espaço pré-processado: **{X_ref.shape[0]:,} × {X_ref.shape[1]}**  \n'.replace(',', '.')
    + f'Distâncias: p05 **{np.percentile(distancias, 5):.2f}** · '
    f'mediana **{np.median(distancias):.2f}** · p95 **{np.percentile(distancias, 95):.2f}**  \n'
    + '`estimate_bandwidth`: '
    + ' · '.join(f'q={q} → **{b:.2f}**' for q, b in bandwidth_ref.items())
))


Espaço pré-processado: **2.350 × 210**  
Distâncias: p05 **3.15** · mediana **4.43** · p95 **5.44**  
`estimate_bandwidth`: q=0.1 → **3.66** · q=0.2 → **3.95** · q=0.3 → **4.16** · q=0.5 → **4.46**

In [9]:
# Métricas de avaliação não supervisionadas (não dependem do rótulo y)
def _get_labels_and_features(estimator, X):
    """Obtém os dados transformados e os rótulos de cluster, tanto para o Pipeline
    completo (preprocessor + modelo) quanto para o modelo isolado (dentro do GridSearchCV)."""
    if isinstance(estimator, Pipeline):
        X_t = estimator[:-1].transform(X)
        labels = estimator[-1].predict(X_t)
    else:
        X_t = X
        labels = estimator.predict(X_t)
    return X_t, labels

def silhouette_scorer(estimator, X, y=None):
    X_t, labels = _get_labels_and_features(estimator, X)
    if len(np.unique(labels)) < 2:
        return -1.0
    return silhouette_score(X_t, labels)

scoring = {
    'silhouette': silhouette_scorer
}

bandwidths = [round(b, 2) for b in bandwidth_ref.values()]

models = [
    ("k-means", KMeans(n_init=10, random_state=42), {'n_clusters': [2, 3, 4, 5, 6, 8], 'init': ['k-means++', 'random']}),
    ("mini batch k-means", MiniBatchKMeans(n_init=10, random_state=42), {'n_clusters': [2, 3, 4, 5, 6, 8], 'batch_size': [64, 128, 256]}),
    ("bisecting k-means", BisectingKMeans(random_state=42), {'n_clusters': [2, 3, 4, 5, 6, 8], 'bisecting_strategy': ['biggest_inertia', 'largest_cluster']}),
    ("birch", Birch(), {'n_clusters': [2, 3, 4, 5, 6, 8], 'threshold': [0.1, 0.3, 0.5], 'branching_factor': [25, 50, 100]}),
    # ("mean shift", MeanShift(), {'bandwidth': bandwidths}),
    # ("gaussian mixture", GaussianMixture(random_state=42, max_iter=1000), {'n_components': [2, 3, 4, 5, 6, 8], 'covariance_type': ['full', 'tied', 'diag', 'spherical']}),
]


In [11]:
def train_model(X, y=None, n_splits=10, final_results = {}):
    for model_name, model_obj, model_params in models:
        inicio = time.time()
        print(f'{model_name} run...', end=' ', flush=True)
        model_gs = GridSearchCV(
            model_obj,
            model_params,
            scoring=silhouette_scorer, # Seleção de hiperparâmetros pela métrica de silhueta
            cv=5
        )

        approach = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model_gs)
        ])
        results = cross_validate(
            approach,
            X=X,
            y=y,
            scoring=scoring,
            cv=ShuffleSplit(n_splits=n_splits, test_size=.2, random_state=42),
            n_jobs=-1
        )
        results["name"] = [model_name] * n_splits
        print(f'{time.time() - inicio:.0f}s')

        if final_results:
            for key, value in results.items():
                final_results[key] = np.append(final_results[key], value)
        else:
            final_results = results
    return final_results


In [ ]:
final_results = train_model(X)

k-means run... 21s
mini batch k-means run... 9s
bisecting k-means run... 5s


In [ ]:
# Métricas em que o menor valor é o melhor (tempos e Davies-Bouldin)
lower_is_better = ['fit_time', 'score_time', 'test_davies_bouldin']

def highlight_max(s, props=''):
    values = [float(value.split()[0]) for value in s.values[1:]]
    result = [''] * len(s.values)
    if s.values[0] in lower_is_better:
        result[np.argmin(values)+1] = props
    else:
        result[np.argmax(values)+1] = props
    return result

def get_winner(s):
    metric = s.values[0]
    values = [float(value.split()[0]) for value in s.values[1:]]
    models = results.columns[1:]

    if s.values[0] in lower_is_better:
        return models[np.argmin(values)]
    else:
        return models[np.argmax(values)]

results = (
    pd
    .DataFrame(final_results)
    .groupby(['name'])
    .agg([lambda x: f"{np.mean(x):.3f} ± {np.std(x):.3f}"])#
    .transpose()
    .reset_index()
    .rename(columns={"level_0": "score"})
    .drop(columns="level_1")
    # .set_index('score')
)
time_scores = ['fit_time', 'score_time']
winner = results.query('score not in @time_scores').apply(get_winner, axis=1).value_counts().index[0]
results.columns.name = ''
results = (
    results
    .style
    .hide(axis='index')
    .apply(highlight_max, props='color:white;background-color:gray', axis=1)
)
display(results)
display(Markdown(f'O melhor modelo é o : **{winner}**'))


score,bisecting k-means,k-means,mini batch k-means
fit_time,4.354 ± 0.172,16.070 ± 0.366,5.674 ± 0.484
score_time,0.038 ± 0.017,0.041 ± 0.016,0.034 ± 0.011
test_silhouette,0.100 ± 0.013,0.104 ± 0.005,0.097 ± 0.012


O melhor modelo é o : **k-means**

In [ ]:
#Obtem o modelo e os parametros ganhadores
model_name, model, model_params  = [foo for foo in models if foo[0] == winner][0]

model_gs = GridSearchCV(model, model_params, scoring=silhouette_scorer)
approach = Pipeline([
    ("preprocessing", preprocessor),
    ("model", model_gs)
])
approach.fit(X) #Seleciona o approach

# Silhueta do modelo final em todo o conjunto de dados
X_t = approach[:-1].transform(X)
labels = approach[-1].predict(X_t)
display(Markdown(f'Silhueta do modelo final (**{model_name}**): **{silhouette_score(X_t, labels):.3f}**'))
display(Markdown(f'Melhores parâmetros: `{approach[-1].best_params_}`'))
display(Markdown(f'Grupos formados: **{len(np.unique(labels))}**'))


Silhueta do modelo final (**k-means**): **0.103**

Melhores parâmetros: `{'init': 'k-means++', 'n_clusters': 3}`

Grupos formados: **3**

In [ ]:
# Mapeando o perfil
perfil = (
    df
    .assign(segmento=labels)
    .groupby('segmento')
    .agg(
        imoveis=('preco_venda', 'size'),
        preco_mediano=('preco_venda', 'median'),
        area_habitavel=('area_habitavel', 'median'),
        qualidade_geral=('qualidade_geral', 'median'),
        idade_imovel=('idade_imovel', 'median'),
    )
    .assign(participacao=lambda d: (d['imoveis'] / len(df)).map('{:.1%}'.format))
    .sort_values('preco_mediano', ascending=False)
)
display(Markdown('### Perfil dos segmentos'))
display(perfil.style.format({'preco_mediano': '${:,.0f}', 'area_habitavel': '{:,.0f}'}))


### Perfil dos segmentos

,imoveis,preco_mediano,area_habitavel,qualidade_geral,idade_imovel,participacao
segmento,,,,,,
1,811,"$214,000","1,661",7.000000,6.000000,34.5%
0,908,"$145,750","1,216",5.000000,42.000000,38.6%
2,631,"$123,000","1,312",5.000000,77.000000,26.9%


In [ ]:
modelo_path = Path(f'../models/model_bonus_imoveis_{winner.replace(" ", "_").replace("-", "_")}.joblib')
joblib.dump(approach, modelo_path) # Salva o modelo em disco
display(Markdown(f'Modelo salvo em `{modelo_path}` ({modelo_path.stat().st_size / 1024:.0f} KB)'))


Modelo salvo em `..\models\model_bonus_imoveis_k_means.joblib` (32 KB)

# Mapeando o perfil dos compradores

In [ ]:
compradores = (
    df
    .assign(segmento=labels)
)

compradores.head()

,ordem,pid,subclasse,zoneamento,area_terreno,tipo_viela,formato_lote,relevo,config_lote,declividade,...,area_varandas,banheiros_total,idade_imovel,anos_desde_reforma,tem_porao,tem_garagem,tem_lareira,tem_segundo_pavimento,preco_por_pe2,segmento
0,2,526350040,20,RH,11622,SemViela,4,Lvl,Inside,3,...,260,1.0,49,49,1,1,0,0,117.187500,0
1,3,526351010,20,RL,14267,SemViela,3,Lvl,Corner,3,...,429,1.5,52,52,1,1,0,0,129.420617,0
2,4,526353030,20,RL,11160,SemViela,4,Lvl,Corner,3,...,0,3.5,42,42,1,1,1,0,115.639810,0
3,5,527105010,60,RL,13830,SemViela,3,Lvl,Inside,3,...,246,2.5,13,12,1,1,1,1,116.574586,1
4,6,527105030,60,RL,9978,SemViela,3,Lvl,Inside,3,...,396,2.5,12,12,1,1,1,1,121.882793,1
